In [1]:
import requests
import json
import psycopg2
from psycopg2.extras import execute_values

/opt/anaconda3/envs/se4g/lib/python3.13/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
# 1. Fetch data from Dati Lombardia API
def fetch_data_from_api(api_url):
    response = requests.get(api_url)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"API request failed with status code {response.status_code}")

In [3]:
# 2. Connect to PostgreSQL database
def connect_to_postgres():
    conn = psycopg2.connect(
        host="localhost",
        database="lombardia_air_quality",
        user="airdata_user",
        password="user"
    )
    return conn

In [4]:
# 3. Create table if it doesn't exist
def create_table_if_not_exists(conn, table_name, data_sample):
    cursor = conn.cursor()
    
    # Create schema SQL statement based on the data structure
    columns = []
    for key, value in data_sample.items():
        column_type = "TEXT"  # Default type
        
        # Try to infer the data type
        if isinstance(value, int):
            column_type = "INTEGER"
        elif isinstance(value, float):
            column_type = "NUMERIC"
        elif isinstance(value, bool):
            column_type = "BOOLEAN"
        elif isinstance(value, dict):
            # Handle location and other complex objects
            if key == "location":
                column_type = "JSONB"  # Store geographical data as JSONB
            else:
                column_type = "JSONB"  # Use JSONB for other nested structures
        # Special case for timestamp fields
        elif key in ["datastart", "datastop"]:
            column_type = "TIMESTAMP"
        # Handle latitude and longitude as numeric
        elif key in ["lat", "lng"]:
            column_type = "NUMERIC"
            
        columns.append(f'"{key}" {column_type}')
    
    create_table_sql = f"""
    DROP TABLE IF EXISTS {table_name};
    CREATE TABLE {table_name} (
        {', '.join(columns)}
    );
    """
    
    cursor.execute(create_table_sql)
    conn.commit()
    cursor.close()
    print(f"Table {table_name} created successfully")

In [5]:
# 4. Prepare data for insertion (handle complex types)
def prepare_data_for_insertion(data_list):
    prepared_data = []
    
    for item in data_list:
        prepared_item = {}
        for key, value in item.items():
            if isinstance(value, dict):
                # Convert dict to JSON string for JSONB fields
                prepared_item[key] = json.dumps(value)
            elif key in ["datastart", "datastop"] and isinstance(value, str):
                # Handle timestamp conversion if needed
                try:
                    # Try to parse the timestamp if it's in a specific format
                    # Adjust the format based on your actual data
                    prepared_item[key] = value
                except:
                    prepared_item[key] = value
            else:
                prepared_item[key] = value
        prepared_data.append(prepared_item)
    
    return prepared_data

In [6]:
# 5. Insert data into table
def insert_data(conn, table_name, data_list):
    cursor = conn.cursor()
    
    if not data_list:
        print("No data to insert")
        return
    
    # Prepare data for insertion
    prepared_data = prepare_data_for_insertion(data_list)
    
    # Get column names from the first data item
    columns = list(prepared_data[0].keys())
    
    # Prepare values for insertion
    values = [[item.get(col) for col in columns] for item in prepared_data]
    
    # Create the SQL query
    insert_query = f"""
    INSERT INTO {table_name} ({', '.join([f'"{col}"' for col in columns])})
    VALUES %s
    ON CONFLICT DO NOTHING
    """
    
    try:
        # Execute the query with all values
        execute_values(cursor, insert_query, values)
        conn.commit()
        print(f"Inserted {len(prepared_data)} records into {table_name}")
    except Exception as e:
        print(f"Error inserting data: {str(e)}")
        conn.rollback()
    finally:
        cursor.close()

In [7]:
# Main execution
def main():
    # API URL
    api_url = "https://www.dati.lombardia.it/resource/ib47-atvt.json"
    # Define the table name for your data
    table_name = "station"
    
    try:
        # Fetch data from API
        print("Fetching data from API...")
        raw_data = fetch_data_from_api(api_url)
        
        # Debug: Inspect the data structure
        print("Sample data item structure:")
        if raw_data:
            print(json.dumps(raw_data[0], indent=2))
            print(f"Total records fetched: {len(raw_data)}")
        
        # Connect to PostgreSQL
        print("Connecting to PostgreSQL...")
        conn = connect_to_postgres()
        
        # Create table if it doesn't exist (using actual data to infer schema)
        print("Creating table if it doesn't exist...")
        if raw_data:
            create_table_if_not_exists(conn, table_name, raw_data[0])
        
        # Insert data into table
        print("Inserting data into table...")
        insert_data(conn, table_name, raw_data)
        
        # Close connection
        conn.close()
        print("Process completed successfully!")
        
    except Exception as e:
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()

# %%
if __name__ == "__main__":
    main()

Fetching data from API...
Sample data item structure:
{
  "idsensore": "12691",
  "nometiposensore": "Arsenico",
  "unitamisura": "ng/m\u00b3",
  "idstazione": "560",
  "nomestazione": "Varese v.Copelli",
  "quota": "383",
  "provincia": "VA",
  "comune": "Varese",
  "storico": "N",
  "datastart": "2008-04-01T00:00:00.000",
  "utm_nord": "5073728",
  "utm_est": "486035",
  "lat": "45.81697450",
  "lng": "8.82024911",
  "location": {
    "type": "Point",
    "coordinates": [
      8.82024911,
      45.8169745
    ]
  },
  ":@computed_region_6hky_swhk": "1",
  ":@computed_region_ttgh_9sm5": "1",
  ":@computed_region_af5v_nc64": "3"
}
Total records fetched: 984
Connecting to PostgreSQL...
Creating table if it doesn't exist...
Table station created successfully
Inserting data into table...
Inserted 984 records into station
Process completed successfully!
